# Práctica 1 · Cierre — Ficha de síntesis del capítulo 1 y expediente de la cuenca
### Hidrología (1CIA-502) · Ingeniería Ambiental · ESPAM MFL · Unidad I
**Docente:** Ing. Cristian Paliz Acosta, MSc.

Este cuaderno **no calcula nada nuevo**: junta lo que ya produjiste y arma dos cosas.

1. La **ficha de síntesis** del capítulo 1 (una página, PNG y PDF), con las mismas filas para todos.
2. El **expediente de tu cuenca** (`expediente_<cuenca>_<estudiante>.csv`), el archivo que crece capítulo a
   capítulo: la Unidad II le añade la precipitación media y la escorrentía, la Unidad III el modelo en HEC-HMS
   y la Unidad IV el agua subterránea. En el proyecto integrador final, ese archivo y los de tus compañeros
   forman el **Compendio hidrológico de cuencas de Manabí**.

**Necesitas:** el ZIP de resultados del paso 2, el del paso 3 y tu hoja de aforo del 21 de octubre.

| Capítulo | Unidad | Cuaderno que alimenta el expediente |
|---|---|---|
| 1 · Ficha morfométrica e hidrométrica | I | `P1_paso2`, `P1_paso3` y este cuaderno |
| 2 · Precipitación y escorrentía | II | Práctica 2 (Thiessen, número de curva) |
| 3 · Modelación de eventos | III | Registro de simulaciones en HEC-HMS |
| 4 · Agua subterránea y flujo base | IV | Borrador del capítulo 4 |

## 1 · CONFIGURACIÓN ✍️ (única celda que editas)
- **AFORO:** copia los valores de tu hoja de aforo tal como quedaron en la hoja (no los rehagas).
- **SINTESIS:** tres frases tuyas, en lenguaje que entendería un agricultor de la cuenca.

In [ ]:
ESTUDIANTE = "Apellidos Nombres"
CUENCA     = "Carrizal"

AFORO = {                                # hoja de aforo con flotadores de tu grupo (21 de octubre)
    "fecha": "2026-10-21",
    "grupo": None,                       # N.º de grupo de la hoja
    "sitio": "Río Carrizal, [sitio de aforo]",
    "longitud_tramo_m": None,            # L, sección 1 de la hoja
    "lanzamientos_validos": None,        # cuántos promediaste (sección 2)
    "velocidad_superficial_m_s": None,   # promedio de los lanzamientos válidos
    "ancho_medio_m": None,               # B, sección 3
    "profundidad_media_m": None,         # sección 3
    "k": 0.85,                           # coeficiente del escenario central (la hoja usa 0,85)
    "area_min_m2": None, "area_central_m2": None, "area_max_m2": None,
    "Q_min_m3_s": None, "Q_central_m3_s": None, "Q_max_m3_s": None,
}

USOS_DEL_AGUA = ""   # p. ej. "abastecimiento de Calceta y Junín; riego del sistema Carrizal-Chone"

SINTESIS = [
    "",   # 1. cómo es la cuenca (forma y relieve)
    "",   # 2. cómo responde a una lluvia intensa
    "",   # 3. cuánta agua hay y qué tan segura es en estiaje
]

## 2 · Subir los resultados de los pasos 2 y 3
Sube los dos ZIP (o directamente los dos `expediente_P2_…csv` y `expediente_P3_…csv`).

In [ ]:
import os, io, glob, zipfile, unicodedata, textwrap
import numpy as np, pandas as pd, matplotlib.pyplot as plt
SLUG = unicodedata.normalize("NFKD", ESTUDIANTE).encode("ascii", "ignore").decode().replace(" ", "_")
os.makedirs("salidas", exist_ok=True); os.makedirs("entradas", exist_ok=True)

try:
    from google.colab import files
    for nombre, contenido in files.upload().items():
        open(os.path.join("entradas", nombre), "wb").write(contenido)
except ImportError:
    print("Fuera de Colab: se leen los archivos que ya estén en la carpeta 'entradas/'.")

for z in glob.glob("entradas/*.zip"):
    with zipfile.ZipFile(z) as zf:
        for n in zf.namelist():
            if os.path.basename(n).startswith("expediente_P"):
                open(os.path.join("entradas", os.path.basename(n)), "wb").write(zf.read(n))

def leer(paso):
    cand = sorted(glob.glob(f"entradas/expediente_{paso}_*.csv"))
    if not cand:
        print(f"❌ Falta el expediente del {paso}. Vuelve a ejecutar ese cuaderno (versión actual del repositorio) y sube su ZIP.")
        return pd.DataFrame(columns=["capitulo", "apartado", "campo", "valor", "unidad", "fuente"])
    print(f"✅ {paso}: {os.path.basename(cand[-1])}")
    return pd.read_csv(cand[-1], dtype=str).fillna("")
P2, P3 = leer("P2"), leer("P3")

## 3 · Unir el expediente y revisar que esté completo
El cuaderno añade tu aforo y tus frases, revisa que no falte ningún campo y hace dos comprobaciones de
coherencia: el Kc recalculado desde A y P, y la incertidumbre del aforo.

In [ ]:
fa = "hoja de aforo · 21 oct · flotadores"
fila_aforo = [
    ("1.5", "Fecha del aforo", AFORO["fecha"], "", fa),
    ("1.5", "Grupo", AFORO["grupo"], "", fa),
    ("1.5", "Sitio del aforo", AFORO["sitio"], "", fa),
    ("1.5", "Longitud del tramo L", AFORO["longitud_tramo_m"], "m", fa),
    ("1.5", "Lanzamientos válidos", AFORO["lanzamientos_validos"], "n.º", fa),
    ("1.5", "Ancho medio B", AFORO["ancho_medio_m"], "m", fa),
    ("1.5", "Profundidad media", AFORO["profundidad_media_m"], "m", fa),
    ("1.5", "Velocidad superficial media", AFORO["velocidad_superficial_m_s"], "m/s", fa),
    ("1.5", "Coeficiente k", AFORO["k"], "—", fa),
    ("1.5", "Área mínima", AFORO["area_min_m2"], "m²", fa),
    ("1.5", "Área central", AFORO["area_central_m2"], "m²", fa),
    ("1.5", "Área máxima", AFORO["area_max_m2"], "m²", fa),
    ("1.5", "Caudal aforado mínimo", AFORO["Q_min_m3_s"], "m³/s", fa),
    ("1.5", "Caudal aforado central", AFORO["Q_central_m3_s"], "m³/s", fa),
    ("1.5", "Caudal aforado máximo", AFORO["Q_max_m3_s"], "m³/s", fa),
    ("1.1", "Usos del agua", USOS_DEL_AGUA, "", "estudiante"),
] + [("1.7", f"Síntesis {i + 1}", t, "", "estudiante") for i, t in enumerate(SINTESIS)]
EXTRA = pd.DataFrame([("1",) + tuple("" if v is None else str(v) for v in f[:3]) + f[3:] for f in fila_aforo],
                     columns=["capitulo", "apartado", "campo", "valor", "unidad", "fuente"])
EXP = pd.concat([P2, P3, EXTRA], ignore_index=True)

faltan = EXP[(EXP.valor.astype(str).str.strip() == "") & ~EXP.campo.isin(["IoU con el límite oficial"])]
if len(faltan):
    print("⚠️  Campos vacíos (complétalos en la celda 1 o vuelve a ejecutar el paso que los produce):")
    for _, r in faltan.iterrows(): print(f"   · {r.apartado} {r.campo}")
else:
    print("✅ Expediente completo")

def val(campo):
    v = EXP.loc[EXP.campo == campo, "valor"]
    try: return float(str(v.iloc[0]).replace(",", ".")) if len(v) else np.nan
    except ValueError: return np.nan

A_, P_, Kc_ = val("Área (A)"), val("Perímetro (P)"), val("Coeficiente de compacidad (Kc)")
if np.isfinite(A_) and np.isfinite(Kc_):
    Kc_rec = 0.28 * P_ / np.sqrt(A_)
    print((f"Kc recalculado = 0,28 × {P_:.1f} / √{A_:.1f} = {Kc_rec:.2f} · en la tabla: {Kc_:.2f}").replace(".", ",")
          + ("  ✅" if abs(Kc_rec - Kc_) < 0.015 else "  ⚠️ no coinciden"))
qmin, qc, qmax = val("Caudal aforado mínimo"), val("Caudal aforado central"), val("Caudal aforado máximo")
if np.isfinite(qc) and qc > 0:
    inc = 100 * (qmax - qmin) / 2 / qc
    print(f"Incertidumbre del aforo: ±{inc:.0f} % alrededor de {qc:.2f} m³/s".replace(".", ",")
          + ("" if qmin <= qc <= qmax else "  ⚠️ el central no está entre el mínimo y el máximo"))
    EXP.loc[len(EXP)] = ["1", "1.5", "Incertidumbre del aforo", f"{inc:.0f}", "± %", "calculada de la hoja"]
    # Mano antes que modelo: el cuaderno rehace las tres cuentas de la sección 4 de la hoja
    v_, k_, ac, amin, amax = (val("Velocidad superficial media"), val("Coeficiente k"), val("Área central"),
                              val("Área mínima"), val("Área máxima"))
    if all(np.isfinite([v_, k_, ac, amin, amax])):
        for nombre, q_hoja, q_cal in [("central", qc, k_ * v_ * ac), ("mínimo", qmin, 0.80 * v_ * amin),
                                      ("máximo", qmax, 0.90 * v_ * amax)]:
            ok = abs(q_cal - q_hoja) <= 0.03 * q_cal
            print(f"   Q {nombre}: hoja {q_hoja:.3f} · recalculado {q_cal:.3f} m³/s".replace(".", ",")
                  + ("  ✅" if ok else "  ⚠️ revisa la cuenta en la hoja (difiere más de 3 %)"))
    B_, d_ = val("Ancho medio B"), val("Profundidad media")
    if np.isfinite(B_) and np.isfinite(d_) and np.isfinite(ac) and abs(B_ * d_ - ac) > 0.15 * ac:
        print(f"   ⚠️ B × profundidad media = {B_ * d_:.2f} m² y el área central es {ac:.2f} m²: si usaste trapecios, explícalo en 1.5".replace(".", ","))
r = val("Q90 / Q50")
if np.isfinite(r):
    print(f"Q90/Q50 = {r:.2f}".replace(".", ",") + " → régimen " + ("regular" if r >= 0.5 else "estacional" if r >= 0.2 else "muy estacional (casi se seca)"))
if "ejemplo" in " ".join(P3.valor.astype(str)):
    print("⚠️  Las curvas del apartado 1.6 salen de los datos de ejemplo del curso: decláralo en el capítulo.")

EXP.to_csv(f"salidas/expediente_{CUENCA}_{SLUG}.csv", index=False, encoding="utf-8-sig")
EXP

## 4 · La ficha de síntesis (una página)
Es la página que se suma al Compendio. Va al final de tu capítulo 1.

In [ ]:
def v(campo, d=None):
    s = EXP.loc[EXP.campo == campo, "valor"]
    s = str(s.iloc[0]) if len(s) else ""
    if d is not None:
        try: s = f"{float(s.replace(',', '.')):,.{d}f}".replace(",", " ").replace(".", ",")
        except ValueError: pass
    return s or "—"

filas = [
 ("Cuenca · punto de salida (UTM 17S)", f"{v('Cuenca')} · E {v('Exutorio Este', 0)} · N {v('Exutorio Norte', 0)}", "m", "1.1"),
 ("Usos del agua", v("Usos del agua"), "", "1.1"),
 ("Área / perímetro / longitud del cauce", f"{v('Área (A)', 1)} / {v('Perímetro (P)', 1)} / {v('Longitud del cauce principal (L)', 2)}", "km² / km / km", "1.3"),
 ("Kc / Ff / IH", f"{v('Coeficiente de compacidad (Kc)', 2)} / {v('Factor de forma (Ff)', 3)} / {v('Integral hipsométrica (IH)', 2)}", "—", "1.3"),
 ("Pendiente del cauce / tc (Kirpich)", f"{v('Pendiente del cauce principal (S)', 4)} / {v('Tiempo de concentración (Kirpich) (tc)', 1)}", "m/m / h", "1.3"),
 ("Cotas mín. / media / máx.", f"{v('Cota mínima (zmín)', 0)} / {v('Cota media (z̄)', 0)} / {v('Cota máxima (zmáx)', 0)}", "m", "1.3"),
 ("Estación de referencia · P anual · alertas", f"{v('Estación de referencia')} · {v('P anual media (climatología 1964–2018)', 0)} · {v('Alertas de calidad', 0)}", "— / mm / n.º", "1.4"),
 ("Aforo: L · lanzamientos válidos · v sup.", f"{v('Longitud del tramo L', 1)} · {v('Lanzamientos válidos', 0)} · {v('Velocidad superficial media', 3)}", "m · n.º · m/s", "1.5"),
 ("Caudal aforado (mín – central – máx)", f"{v('Caudal aforado mínimo', 2)} – {v('Caudal aforado central', 2)} – {v('Caudal aforado máximo', 2)} (±{v('Incertidumbre del aforo')} %)", "m³/s", "1.5"),
 ("Curva de gasto a / h₀ / b / R²", f"{v('Curva de gasto a', 3)} / {v('Curva de gasto h0', 3)} / {v('Curva de gasto b', 3)} / {v('R² del ajuste', 4)}", "— / m / — / —", "1.6"),
 ("Caudal medio / Q50 / Q90 · Q90÷Q50", f"{v('Caudal medio', 2)} / {v('Q50', 2)} / {v('Q90', 2)} · {v('Q90 / Q50', 2)}", "m³/s · —", "1.6"),
 ("Fuente de los caudales", v("Fuente de los datos de caudal"), "", "1.6"),
]
fig = plt.figure(figsize=(8.27, 11.69))
ax = fig.add_axes([0.07, 0.04, 0.86, 0.92]); ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.autoscale(False)
ax.text(0, 1.0, f"Ficha de síntesis · Cuenca {v('Cuenca')}", fontsize=17, fontweight="bold", va="top", color="#12263A")
ax.text(0, 0.965, f"Capítulo 1 · Hidrología 1CIA-502 · ESPAM MFL · {ESTUDIANTE}", fontsize=10, va="top", color="#44525E")
y = 0.92
ax.plot([0, 1], [y + 0.008, y + 0.008], color="#12263A", lw=1.2)
for campo, valor, unidad, ap in filas:
    lin = textwrap.wrap(valor, 40) or ["—"]
    ax.text(0, y, campo, fontsize=9.5, fontweight="bold", va="top", color="#12263A")
    for i, l in enumerate(lin):
        ax.text(0.43, y - i * 0.018, l, fontsize=9.5, va="top")
    ax.text(0.80, y, unidad, fontsize=8, va="top", color="#5B6B78"); ax.text(1.0, y, ap, fontsize=8.5, va="top", ha="right", color="#B3541A")
    y -= 0.022 + 0.018 * len(lin)
    ax.plot([0, 1], [y + 0.012, y + 0.012], color="#D9D3C4", lw=0.6)
y -= 0.01
ax.text(0, y, "Cómo es y cómo responde la cuenca", fontsize=11, fontweight="bold", va="top", color="#12263A"); y -= 0.03
for i, t in enumerate(SINTESIS):
    for j, l in enumerate(textwrap.wrap(f"{i + 1}. {t or '—'}", 95)):
        ax.text(0, y, l, fontsize=9.5, va="top"); y -= 0.019
    y -= 0.008
ax.text(0, 0.0, "Fuente: expediente de la cuenca (P1_paso2, P1_paso3, hoja de aforo). MDE ALOS PALSAR; climatología INAMHI 1964–2018. "
        "Cuadernos CC BY-NC-SA 4.0 · github.com/crispaliz/hidrologia-espam", fontsize=7, color="#5B6B78", va="bottom", wrap=True)
for ext in ("png", "pdf"):
    plt.savefig(f"salidas/ficha_sintesis_{CUENCA}_{SLUG}.{ext}", dpi=200 if ext == "png" else None)
plt.show()

## 5 · Descargar

In [ ]:
import shutil
paquete = shutil.make_archive(f"Capitulo1_{CUENCA}_{SLUG}", "zip", "salidas")
print("✅ Paquete:", paquete)
try:
    from google.colab import files; files.download(paquete)
except ImportError:
    print("Descárgalo desde el panel de archivos de Colab.")

---
**Guarda `expediente_<cuenca>_<estudiante>.csv`.** Es la entrada del primer cuaderno de la Unidad II y no se
vuelve a generar desde cero: cada capítulo le añade sus filas.